# Airbnb Price Analytics — Prediction

**Region:** United Kingdom
**Currency:** GBP

This notebook scores one city with a frozen model version from `03_price_model`. It does not fit anything: the model, baseline statistics and category lists are loaded from the freeze package.

The same notebook is used for London and Bristol (development cities) and for a new city (holdout). Only the `city` parameter changes.

## 1. Configuration

The city, model version, catalog, schemas and model location are notebook parameters.

Writing to Delta tables is disabled by default during testing.

In [ ]:
import re, json, hashlib, uuid
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn, joblib
from pyspark.sql import functions as F
from sklearn.base import BaseEstimator, TransformerMixin
dbutils.widgets.text('catalog', 'workspace')
dbutils.widgets.text('gold_schema', 'airbnb_gold')
dbutils.widgets.text('silver_schema', 'airbnb_silver')
dbutils.widgets.text('city', 'london')
dbutils.widgets.text('model_version', 'v1')
dbutils.widgets.text('model_root', '/Volumes/workspace/airbnb_eda/artifacts/models')
dbutils.widgets.dropdown('write_tables', 'false', ['false', 'true'])
RUN_LABELS = ['first_run', 'after_pipeline_fix', 'after_model_change']
dbutils.widgets.dropdown('run_label', 'first_run', RUN_LABELS)

Widget values are read once and normalised. City names are lower case, as in the ETL pipeline. `run_label` applies only to a holdout city and identifies the first run and any later runs after a classified change; development cities always use `development`.

In [ ]:
catalog = dbutils.widgets.get('catalog').strip()
gold_schema = dbutils.widgets.get('gold_schema').strip()
silver_schema = dbutils.widgets.get('silver_schema').strip()
city = dbutils.widgets.get('city').strip().lower()
model_version = dbutils.widgets.get('model_version').strip()
model_root = dbutils.widgets.get('model_root').rstrip('/')
write_tables = dbutils.widgets.get('write_tables') == 'true'
run_label = dbutils.widgets.get('run_label').strip()

Identifiers are validated before they are used in table names, SQL predicates and paths.

In [ ]:
for name, value in [('catalog', catalog), ('gold_schema', gold_schema), ('silver_schema', silver_schema)]:
    if not re.fullmatch('[A-Za-z][A-Za-z0-9_]*', value):
        raise ValueError(f'Invalid {name} identifier')
if not re.fullmatch('[a-z][a-z0-9_-]*', city):
    raise ValueError(f'Invalid city: {city!r}')
if not re.fullmatch('[A-Za-z0-9][A-Za-z0-9._-]*', model_version):
    raise ValueError('model_version must be a plain folder name')
if not re.fullmatch(f'/Volumes/{catalog}/[A-Za-z][A-Za-z0-9_]*/[A-Za-z][A-Za-z0-9_]*(/.*)?', model_root):
    raise ValueError('model_root must be a Volume path in the selected catalog')
if run_label not in RUN_LABELS:
    raise ValueError(f'run_label must be one of {RUN_LABELS}')

Table names and the freeze package location.

In [ ]:
KEY_COLUMNS = ['city', 'listing_id', 'snapshot_date']
features_table = f'{catalog}.{gold_schema}.features'
predictions_table = f'{catalog}.{gold_schema}.predictions'
metrics_table = f'{catalog}.{gold_schema}.model_metrics'
silver_table = f'{catalog}.{silver_schema}.listings'
artifact_dir = Path(model_root) / model_version
print(f'city={city}, model_version={model_version}, artifact_dir={artifact_dir}, write_tables={write_tables}')

`spark_ready` prepares a pandas DataFrame for Spark: NaN becomes `None`, so missing values are stored as NULL rather than NaN in nullable columns.

In [ ]:
def spark_ready(frame):
    return frame.astype(object).where(frame.notna(), None)

## 2. Shared Code From the Training Notebook

The next two cells are identical copies of the cells in `03_price_model`. The saved model refers to `RareCategoryGrouper` by name, so the class must be defined before the model is loaded.

Change these cells only together with `03_price_model` and a new model version. The check set in section 3 stops the notebook if their behaviour differs from the frozen package.

In [ ]:
class RareCategoryGrouper(BaseEstimator, TransformerMixin):
    """Replace rare, unseen and missing category values with 'Other'."""

    def __init__(self, columns, min_count):
        self.columns = columns
        self.min_count = min_count

    def fit(self, X, y=None):
        # Allowed categories are learned from the fitting rows only.
        self.allowed_ = {c: sorted(X[c].value_counts().loc[lambda s: s >= self.min_count].index) for c in self.columns}
        return self

    def transform(self, X):
        X = X.copy()
        for c in self.columns:
            X[c] = X[c].where(X[c].isin(self.allowed_[c]), 'Other').astype(object)
        return X

`predict_baseline` applies the saved baseline statistics with the fixed fallbacks: group median, then room-type median, then global median.

In [ ]:
def predict_baseline(df, stats):
    keys = df[['room_type', 'accommodates']].astype({'accommodates': 'Int64'})
    group = pd.DataFrame(stats['group_medians'], columns=['room_type', 'accommodates', 'median_price_gbp', 'listings'])
    room = pd.DataFrame(stats['room_type_medians'], columns=['room_type', 'median_price_gbp', 'listings'])
    group_price = keys.merge(group.astype({'accommodates': 'Int64'}), how='left', on=['room_type', 'accommodates'],
                             validate='many_to_one')['median_price_gbp'].to_numpy(dtype=float)
    room_price = keys[['room_type']].merge(room, how='left', on='room_type',
                                           validate='many_to_one')['median_price_gbp'].to_numpy(dtype=float)
    has_group, has_room = ~np.isnan(group_price), ~np.isnan(room_price)
    out = df.copy()
    out['baseline_price_gbp'] = np.select([has_group, has_room], [group_price, room_price], stats['global_median_price_gbp'])
    out['baseline_level'] = np.select([has_group, has_room], ['group', 'room_type'], 'global')
    return out

## 3. Freeze Package

The model version must have a complete freeze package. The manifest is read first; every other file must match the checksum recorded in it.

In [ ]:
manifest_path = artifact_dir / 'manifest.json'
if not manifest_path.is_file():
    raise FileNotFoundError(f'No frozen package for model_version={model_version}: {manifest_path} not found')
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
if manifest['model_version'] != model_version:
    raise ValueError(f"Manifest belongs to {manifest['model_version']}, not {model_version}")

Any changed or missing file stops the notebook.

In [ ]:
def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

changed_files = [name for name, expected in manifest['files'].items()
                 if not (artifact_dir / name).is_file() or sha256_file(artifact_dir / name) != expected]
if changed_files:
    raise ValueError(f'Freeze package files missing or changed: {changed_files}')
print('Checksums verified:', sorted(manifest['files']))

The model, baseline statistics, check set and feature allowlist are loaded from the package.

In [ ]:
model = joblib.load(artifact_dir / 'model.joblib')
baseline_stats = json.loads((artifact_dir / 'baseline_stats.json').read_text(encoding='utf-8'))
check_set = json.loads((artifact_dir / 'check_set.json').read_text(encoding='utf-8'))
CATEGORICAL, NUMERIC = manifest['features']['categorical'], manifest['features']['numeric']
FEATURES = CATEGORICAL + NUMERIC
print(f"model={manifest['model_name']}, feature_version={manifest['feature_version']}, features={FEATURES}")

Package versions are compared with the frozen environment. A difference is only a warning: the check set below decides whether the predictions are reproduced.

In [ ]:
current_environment = {'sklearn': sklearn.__version__, 'pandas': pd.__version__,
                       'numpy': np.__version__, 'joblib': joblib.__version__}
for package, version in current_environment.items():
    frozen = manifest['environment'].get(package)
    if version != frozen:
        print(f'WARNING: {package} {version} differs from the frozen environment ({frozen})')

The check set is recomputed with this notebook's code and environment. If any baseline level, baseline price or model prediction differs from the frozen values, the notebook stops.

In [ ]:
check_rows = pd.DataFrame(check_set['rows'])
check_baseline = predict_baseline(check_rows, baseline_stats)
check_model = model.predict(check_rows[FEATURES])
tolerance = check_set['tolerance']
baseline_differs = (~np.isclose(check_baseline['baseline_price_gbp'], check_rows['baseline_price_gbp'], rtol=tolerance)
                    | (check_baseline['baseline_level'] != check_rows['baseline_level']))
model_differs = ~np.isclose(check_model, check_rows['predicted_price_gbp'], rtol=tolerance)
if baseline_differs.any() or model_differs.any():
    raise ValueError(f'Check set failed: {int(baseline_differs.sum())} baseline and {int(model_differs.sum())} '
                     'model predictions differ from the freeze package')
print(f'Check set verified: {len(check_rows)} rows')

## 4. Development or Holdout City

A city listed in the manifest's training cities is a development city: its test rows are evaluated with `dataset = 'development'` and its predictions use `run_label = 'development'`. Any other city is a holdout city: all its rows are labelled `holdout`, its predictions carry the selected `run_label`, and its results are recorded as `dataset = 'city_c_<run_label>'` (for example `city_c_first_run`).

A holdout city is scored only when `write_tables=true`, so every run is recorded. Otherwise the notebook stops here, before reading any of its data.

In [ ]:
is_development = city in manifest['training_cities']
prediction_run_label = 'development' if is_development else run_label
dataset = 'development' if is_development else f'city_c_{run_label}'
eval_split = 'test' if is_development else 'holdout'
if not is_development and not write_tables:
    print(f'Stopped before reading {city}: a holdout city is scored only with write_tables=true, '
          'so that the run is recorded.')
    dbutils.notebook.exit('Stopped before reading holdout data')
print(f"{city}: {'development' if is_development else 'holdout'} city, run_label={prediction_run_label}, "
      f"dataset={dataset}, evaluated split={eval_split}")

A holdout city has exactly one first run, across all model versions, and it is never overwritten. Metrics are written before predictions, so a first run is complete when its predictions exist.

`existing_first_run` returns the model versions with first-run metrics for this city and whether first-run predictions exist.

In [ ]:
def existing_first_run():
    versions, complete = set(), False
    if spark.catalog.tableExists(metrics_table):
        versions = {r['model_version'] for r in spark.table(metrics_table)
                    .filter(f"city = '{city}' AND dataset = 'city_c_first_run'").select('model_version').distinct().collect()}
    if spark.catalog.tableExists(predictions_table):
        complete = spark.table(predictions_table).filter(f"city = '{city}' AND run_label = 'first_run'").limit(1).count() > 0
    return versions, complete

The rules are checked before any data of the city is read:

- `first_run` stops if a complete first run exists, or if first-run metrics belong to another model version. If only this version's metrics exist (an interrupted first run), the run completes the predictions and keeps the metrics.
- `after_pipeline_fix` and `after_model_change` stop if no complete first run exists.

In [ ]:
if not is_development:
    first_run_versions, first_run_complete = existing_first_run()
    if run_label == 'first_run' and first_run_complete:
        raise ValueError(f'The first run of {city} is already recorded; use run_label after_pipeline_fix or after_model_change')
    if run_label == 'first_run' and first_run_versions - {model_version}:
        raise ValueError(f'The first run of {city} belongs to model version(s) {sorted(first_run_versions)}, not {model_version}')
    if run_label != 'first_run' and not first_run_complete:
        raise ValueError(f'No complete first run for {city}; record run_label=first_run before {run_label}')
    if run_label == 'first_run' and first_run_versions:
        print(f'First-run metrics for {city} exist without predictions: completing the interrupted first run')

When writing is enabled, a tiny temporary Delta table with a unique name is created in the Gold schema and dropped again before any rows are read. A missing privilege therefore stops the run before scoring, not halfway through writing. This notebook only reads the freeze package, so it needs no write access to the model Volume.

In [ ]:
def check_table_privilege():
    probe_table = f'{catalog}.{gold_schema}.tmp_write_check_{uuid.uuid4().hex[:12]}'
    try:
        spark.createDataFrame([(1,)], 'probe int').write.format('delta').saveAsTable(probe_table)
    except Exception as error:
        raise PermissionError(f'Cannot create tables in {catalog}.{gold_schema}; USE SCHEMA and CREATE TABLE '
                              f'privileges are required. Spark error: {error}') from error
    spark.sql(f'DROP TABLE IF EXISTS {probe_table}')

if write_tables:
    check_table_privilege()
    print(f'Table creation in {catalog}.{gold_schema} verified')
else:
    print('Table privilege check skipped: dry run')

## 5. Load Features

We read every feature row of the selected city. The model needs the keys, target, feature version, split and every allowlisted feature; if any of these columns is missing, the notebook stops.

The first line repeats the holdout gate, so this cell cannot read holdout rows without `write_tables=true` even when run on its own.

In [ ]:
assert is_development or write_tables, 'Holdout rows are read only when write_tables=true'
features_all = spark.table(features_table)
required_columns = KEY_COLUMNS + ['price_gbp', 'target_valid', 'feature_version', 'split'] + FEATURES
missing_columns = [c for c in required_columns if c not in features_all.columns]
if missing_columns:
    raise ValueError(f'{features_table} is missing required columns {missing_columns}; '
                     f'model {model_version} cannot score it without a pipeline fix')
city_pdf = features_all.filter(F.col('city') == city).select(*required_columns).toPandas()
if city_pdf.empty:
    raise ValueError(f'No rows for city={city} in {features_table}')

`pipeline_version` and the source checksum are joined from Silver on the listing keys. Each feature row must match at most one Silver row. The checksum identifies the input file in the run record.

In [ ]:
missing_silver = [c for c in ['pipeline_version', 'source_sha256'] if c not in spark.table(silver_table).columns]
if missing_silver:
    raise ValueError(f'{silver_table} is missing columns {missing_silver}')
silver_pdf = (spark.table(silver_table).filter(F.col('city') == city)
              .select(*KEY_COLUMNS, 'pipeline_version', 'source_sha256').toPandas())
city_pdf = city_pdf.merge(silver_pdf, on=KEY_COLUMNS, how='left', validate='one_to_one')

Rows without a Silver match keep a null `pipeline_version`, and the city is expected to come from one source file.

In [ ]:
missing_pipeline = int(city_pdf['pipeline_version'].isna().sum())
if missing_pipeline:
    print(f'WARNING: {missing_pipeline} feature rows have no matching Silver row; pipeline_version is null')
input_checksums = sorted(set(city_pdf['source_sha256'].dropna()))
if len(input_checksums) != 1:
    print(f'WARNING: expected one source checksum for {city}, found {len(input_checksums)}')

The city must have one snapshot and the feature version the model was trained on.

In [ ]:
snapshots = sorted(city_pdf['snapshot_date'].unique())
if len(snapshots) != 1:
    raise ValueError(f'Expected one snapshot for {city}, found {snapshots}')
snapshot_date = snapshots[0]
feature_versions = set(city_pdf['feature_version'])
if feature_versions != {manifest['feature_version']}:
    raise ValueError(f"Feature version {feature_versions} differs from the model's {manifest['feature_version']}")

Rows are sorted by key so that results are reproducible. Numeric features are converted to floats, so an entirely missing column becomes NaN rather than empty objects. Values are unchanged, so this matches training.

In [ ]:
city_pdf = city_pdf.sort_values(KEY_COLUMNS).reset_index(drop=True)
city_pdf[NUMERIC] = city_pdf[NUMERIC].astype('float64')
city_pdf['target_valid'] = city_pdf['target_valid'].fillna(False).astype(bool)
print(f'Loaded {len(city_pdf)} rows for {city}, snapshot {snapshot_date}')

## 6. Split Labels

Development cities use the `split` values saved by `03_price_model`; every row must have one. A holdout city may only have empty or `holdout` split values (from an earlier run); any other value stops the notebook. All its rows are labelled `holdout`.

In [ ]:
missing_split = int(city_pdf['split'].isna().sum())
other_labels = sorted(set(city_pdf['split'].dropna()) - {'holdout'})
if is_development and missing_split:
    raise ValueError(f'{missing_split} {city} rows have no split; run 03_price_model first')
if not is_development and other_labels:
    raise ValueError(f'Holdout city {city} has split values {other_labels}; expected none or holdout')
if not is_development:
    city_pdf['split'] = 'holdout'
display(city_pdf.groupby('split').agg(listings=('listing_id', 'size'), valid_targets=('target_valid', 'sum')).reset_index())

For a holdout city, `holdout` is also written into the empty `split` values of `workspace.airbnb_gold.features` with `MERGE`, so the feature table records the assignment as the data contract requires. Existing values are never changed.

In [ ]:
def mark_holdout_split():
    keys = spark.createDataFrame(spark_ready(city_pdf[KEY_COLUMNS]), schema='city string, listing_id string, snapshot_date date')
    keys.createOrReplaceTempView('holdout_keys')
    spark.sql(f'''
        MERGE INTO {features_table} AS t USING holdout_keys AS s
        ON t.city = s.city AND t.listing_id = s.listing_id AND t.snapshot_date = s.snapshot_date
        WHEN MATCHED AND t.split IS NULL THEN UPDATE SET t.split = 'holdout' ''')

if not is_development:
    assert write_tables, 'Holdout rows are written only when write_tables=true'
    mark_holdout_split()
    print(f'features.split set to holdout for empty {city} rows')

## 7. Scoring

Every row is scored by the baseline and the model, including rows without a valid price. Unseen or missing categories become `Other` and missing numbers are handled inside the model pipeline, exactly as in training.

A prediction is `valid` when it is a finite, non-negative price. `actual_price_gbp` is the price only where the target is valid.

In [ ]:
scored = predict_baseline(city_pdf, baseline_stats)
scored['predicted_price_gbp'] = model.predict(scored[FEATURES])
valid_prediction = np.isfinite(scored['predicted_price_gbp']) & (scored['predicted_price_gbp'] >= 0)
scored['prediction_status'] = np.where(valid_prediction, 'valid', 'invalid')
scored['actual_price_gbp'] = scored['price_gbp'].where(scored['target_valid'])
print(f'Prediction coverage: {valid_prediction.mean():.2%} of {len(scored)} listings')

### 7.1 Unseen Categories and Missing Values

The model's own grouping step shows how many listings were mapped to `Other`. Values outside the model's category lists produce a warning, not an error.

In [ ]:
grouped = model.regressor_.named_steps['group'].transform(scored[FEATURES])
display(pd.DataFrame([{'feature': c, 'other_listings': int((grouped[c] == 'Other').sum()),
                       'missing_listings': int(scored[c].isna().sum())} for c in CATEGORICAL]))
for c in CATEGORICAL:
    outside = scored.loc[(grouped[c] == 'Other') & scored[c].notna(), c].value_counts()
    if len(outside):
        print(f'WARNING: {outside.sum()} {c} values outside the model categories were mapped to Other, '
              f'e.g. {outside.head(5).to_dict()}')

Missing numeric values per feature. A feature that is missing for every listing still produces predictions, but they rely on imputed or default values for that feature.

In [ ]:
missing_numeric = scored[NUMERIC].isna().sum()
display(missing_numeric.rename_axis('feature').reset_index(name='missing_listings'))
entirely_missing = list(missing_numeric[missing_numeric == len(scored)].index)
if entirely_missing:
    print(f'WARNING: features missing for every {city} listing: {entirely_missing}')

Baseline fallback levels used for this city.

In [ ]:
display(scored['baseline_level'].value_counts().reindex(baseline_stats['levels'], fill_value=0)
        .rename_axis('baseline_level').reset_index(name='listings'))

## 8. Metrics

Metrics use the evaluated split (`test` for a development city, `holdout` for a holdout city) and only rows with a valid target (`eligible_listings`, the coverage denominator). Coverage is the share of these rows with a valid prediction; MAE and median absolute error use the covered rows (`evaluated_listings`). Values are rounded as in `03_price_model`.

In [ ]:
def evaluate(frame, prediction_col, predictor):
    covered = np.isfinite(frame[prediction_col]) & (frame[prediction_col] >= 0)
    errors = (frame['price_gbp'] - frame[prediction_col])[covered].abs()
    return {'predictor': predictor, 'evaluated_listings': int(covered.sum()), 'eligible_listings': len(frame),
            'mae_gbp': round(errors.mean(), 2), 'median_ae_gbp': round(errors.median(), 2),
            'prediction_coverage': round(covered.mean(), 4) if len(frame) else np.nan}

Baseline and model metrics for the selected city.

In [ ]:
eligible = scored[(scored['split'] == eval_split) & scored['target_valid']]
if eligible.empty:
    print(f'WARNING: no {eval_split} rows with a valid target for {city}; MAE is unavailable')
city_metrics = pd.DataFrame([evaluate(eligible, 'baseline_price_gbp', 'baseline'),
                             evaluate(eligible, 'predicted_price_gbp', 'model')])
display(city_metrics)

The metrics are reshaped to the `model_metrics` format used by `03_price_model`: one row per predictor and metric. A holdout run also records the run itself: the number of rows loaded and the checksum of the input file.

In [ ]:
metrics_pdf = (city_metrics.melt(id_vars=['predictor', 'evaluated_listings', 'eligible_listings'], var_name='metric')
               .assign(model_version=model_version, model_name=manifest['model_name'],
                       feature_version=manifest['feature_version'], dataset=dataset, split=eval_split, city=city,
                       rows_loaded=None if is_development else len(city_pdf),
                       source_sha256=None if is_development else ','.join(input_checksums)))
metrics_columns = ['model_version', 'model_name', 'feature_version', 'dataset', 'split', 'city', 'predictor', 'metric',
                   'value', 'evaluated_listings', 'eligible_listings', 'rows_loaded', 'source_sha256']
print(f'Run record: rows_loaded={len(city_pdf)}, eligible_listings={len(eligible)}, source_sha256={input_checksums}')

## 9. Saving Metrics

Metrics are saved before predictions, so a run that stops halfway always has its metrics recorded. Existing results are never overwritten:

- **Development city:** `03_price_model` already recorded the test metrics. If they exist, the recomputed values must match them (a difference stops the notebook) and nothing is written. If they do not exist, they are appended.
- **Holdout city:** each run label is recorded once. If `city_c_<run_label>` rows already exist for this model version and city, they are kept and the new values are not written.

In [ ]:
metrics_predicate = (f"model_version = '{model_version}' AND dataset = '{dataset}' "
                     f"AND split = '{eval_split}' AND city = '{city}'")
existing = (spark.table(metrics_table).filter(metrics_predicate).toPandas()
            if spark.catalog.tableExists(metrics_table) else pd.DataFrame(columns=metrics_columns))
print(f'Existing metric rows for {city}: {len(existing)}')

For a development city, saved and recomputed metrics are compared, including both listing counts. Values are rounded to two decimals in both notebooks, so a difference of up to 0.01 is allowed.

In [ ]:
def same_metrics(saved, current):
    joined = current.merge(saved, on=['predictor', 'metric'], how='left', suffixes=('', '_saved'))
    return (len(saved) == len(current)
            and np.isclose(joined['value'].astype(float), joined['value_saved'].astype(float), atol=0.01, equal_nan=True).all()
            and (joined['evaluated_listings'] == joined['evaluated_listings_saved']).all()
            and (joined['eligible_listings'] == joined['eligible_listings_saved']).all())

if is_development and len(existing) and not same_metrics(existing, metrics_pdf):
    raise ValueError(f'Recomputed {city} test metrics differ from the saved development metrics')

New metric rows are appended only when none exist for this city, model version and dataset.

In [ ]:
if not write_tables:
    print('DRY RUN: metrics not written')
elif len(existing):
    print(f'Existing {dataset} metrics for {city} kept; nothing written')
else:
    (spark.createDataFrame(spark_ready(metrics_pdf[metrics_columns]), schema=(
         'model_version string, model_name string, feature_version string, dataset string, split string, '
         'city string, predictor string, metric string, value double, evaluated_listings long, '
         'eligible_listings long, rows_loaded long, source_sha256 string'))
     .withColumn('generated_at', F.current_timestamp())
     .write.format('delta').mode('append').saveAsTable(metrics_table))
    print('Written:', metrics_table, dataset, city)

## 10. Saving Predictions

Predictions are written to `workspace.airbnb_gold.predictions` with the columns required by the data contract, plus `baseline_level` and `run_label`. A rerun replaces only the rows of this city, snapshot, model version and run label; first-run predictions are protected by the checks in section 4.

In [ ]:
prediction_columns = ['city', 'listing_id', 'snapshot_date', 'model_version', 'run_label', 'pipeline_version', 'split',
                      'actual_price_gbp', 'predicted_price_gbp', 'baseline_price_gbp', 'baseline_level', 'prediction_status']
predictions_pdf = spark_ready(scored.assign(model_version=model_version, run_label=prediction_run_label)[prediction_columns])
predictions_sdf = (spark.createDataFrame(predictions_pdf, schema=(
                       'city string, listing_id string, snapshot_date date, model_version string, run_label string, '
                       'pipeline_version string, split string, actual_price_gbp double, predicted_price_gbp double, '
                       'baseline_price_gbp double, baseline_level string, prediction_status string'))
                   .withColumn('generated_at', F.current_timestamp()))
predictions_predicate = (f"city = '{city}' AND snapshot_date = DATE '{snapshot_date}' "
                         f"AND model_version = '{model_version}' AND run_label = '{prediction_run_label}'")

The table is created on first use, partitioned like the feature table.

In [ ]:
if write_tables:
    if not spark.catalog.tableExists(predictions_table):
        predictions_sdf.write.format('delta').partitionBy('city', 'snapshot_date').saveAsTable(predictions_table)
    else:
        (predictions_sdf.write.format('delta').mode('overwrite')
         .option('replaceWhere', predictions_predicate).saveAsTable(predictions_table))
    print('Written:', predictions_table, city, snapshot_date)
else:
    print('DRY RUN: nothing written. Set widget write_tables=true only after review.')

## 11. Final Checks

There must be exactly one prediction per feature row. After writing, the stored predictions are counted again, and `actual_price_gbp` must be NULL (not NaN) exactly for the rows without a valid target.

In [ ]:
assert len(scored) == len(city_pdf) and not scored.duplicated(KEY_COLUMNS).any(), 'Expected one prediction per feature row'
if write_tables:
    stored = spark.table(predictions_table).filter(predictions_predicate)
    assert stored.count() == len(scored), 'Stored prediction count differs from the feature rows'
    null_actual = stored.filter(F.col('actual_price_gbp').isNull()).count()
    invalid_targets = int((~scored['target_valid']).sum())
    assert null_actual == invalid_targets, f'{null_actual} NULL actual prices, expected {invalid_targets}'
    print(f'Validated {len(scored)} stored predictions for {city} ({snapshot_date}); {null_actual} without a valid target')
else:
    print(f'DRY RUN validated {len(scored)} predictions in memory')

Prediction coverage and fallback use by split.

In [ ]:
display(scored.assign(valid_prediction=valid_prediction).groupby('split').agg(
    listings=('listing_id', 'size'), valid_targets=('target_valid', 'sum'),
    prediction_coverage=('valid_prediction', 'mean'),
    baseline_fallbacks=('baseline_level', lambda s: int((s != 'group').sum()))).reset_index())